## Questão 6 - Previsão de demanda

### Cenário

O Sr. Almir está furioso. No último verão, o estoque de "Coletes Salva-Vidas" acabou em 3 meses, e a empresa perdeu milhares de reais em vendas. Por outro lado, compraram "Âncoras" demais e elas estão enferrujando no galpão. Gabriel Santos, o Tech Lead, disse que não dá mais para confiar no "feeling". Ele quer um modelo preditivo que diga exatamente quantas unidades venderemos no próximo mês para ajustar as compras com fornecedores.

### Premissas obrigatórias:
- O período de treino deve incluir dados até 31/12/2025.
- O período de teste deve ser o primeiro trimestre de 2026.
- A previsão deve ser feita em base mensal.
- Considere apenas o produto: "Bússola de Bordo 702"

### Tarefa:
1. Utilize os datasets products.csv, product_variants, orders.csv e order_items.csv para criar um dataset unificado que facilite a criação do modelo preditivo.
2. Construa um modelo baseline simples, utilizando: Média móvel dos últimos 3 meses de vendas (considerando apenas dados anteriores à data prevista).
3. Gere a previsão mensal de vendas para o primeiro trimestre de 2026.
4. Compare as previsões com os valores reais do período de teste utilizando a métrica: MAE (Mean Absolute Error)
5. Responda objetivamente:
a) O baseline é adequado para esse produto?
b)  Cite uma limitação desse método.


A construção foi feita como uma sequência de transformação:
arquivos separados → vendas do produto → demanda mensal → previsão → avaliação.


### 1. Importação

In [ ]:
# ============================================================
# ETAPA 1 - Importação das bibliotecas necessárias
# ============================================================
import pandas as pd

pandas foi usado porque facilita leitura de CSVs, relacionamentos entre tabelas, agregações por mês e criação da tabela final de previsões.

### 2.  Leitura dos quatro arquivos obrigatórios

In [ ]:
# ============================================================
# ETAPA 2 - Leitura dos datasets necessários
# ============================================================
products = pd.read_csv("products.csv")
product_variants = pd.read_csv("product_variants.csv")
orders = pd.read_csv("orders.csv")
order_items = pd.read_csv("order_items.csv")


Cada tabela tem uma função:
- `products`: contém o nome do produto, como “Bússola de Bordo 702”.
- `product_variants`: relaciona uma variação ao produto principal.
- `order_items`: informa qual variação foi comprada e quantas unidades (quantity).
- `orders`: informa quando aquele pedido ocorreu (created_at).
Nenhuma tabela isolada possui tudo o que é necessário. Por isso, elas precisam ser unidas.

### 3. Conversão de tipos

In [ ]:
# ============================================================
# ETAPA 3 - Padronização das colunas utilizadas na análise
# ============================================================
orders["created_at"] = pd.to_datetime(orders["created_at"])
order_items["quantity"] = pd.to_numeric(order_items["quantity"])


- `created_at` chegou como texto no CSV. A conversão para data permite extrair mês e ano depois.
- `quantity` precisa ser numérica porque será somada para calcular a demanda mensal.

### 4. Criação do dataset unificado

In [ ]:
# ============================================================
# ETAPA 4 - Criação do dataset unificado
# Cadeia: products -> product_variants -> order_items -> orders
# ============================================================
dataset_unificado = (
    order_items
    .merge(
        product_variants[["id", "product_id"]],
        left_on="product_variant_id",
        right_on="id",
        how="inner",
        suffixes=("_order_item", "_variant")
    )
    .merge(
        products[["id", "name"]],
        left_on="product_id",
        right_on="id",
        how="inner",
        suffixes=("", "_product")
    )
    .merge(
        orders[["id", "created_at"]],
        left_on="order_id",
        right_on="id",
        how="inner",
        suffixes=("", "_order")
    )
)

dataset_unificado = dataset_unificado[
    ["order_id", "product_variant_id", "quantity", "product_id", "name", "created_at"]
]

print("Dataset unificado:")
display(dataset_unificado.head())

Dataset unificado:


,order_id,product_variant_id,quantity,product_id,name,created_at
0,1,113,6,59,Bateria Náutica 8789,2022-09-06 05:37:37
1,2,293,9,146,Cabo Náutico 7323,2023-02-03 04:36:21
2,2,366,2,180,Motor de Popa 1949,2023-02-03 04:36:21
3,2,561,1,275,Cabo Náutico 5921,2023-02-03 04:36:21
4,2,385,5,190,GPS Plotter 3107,2023-02-03 04:36:21


### 5. Filtro do produto-alvo

In [9]:
# ============================================================
# ETAPA 5 - Filtro do produto-alvo
# ============================================================
produto_alvo = "Bússola de Bordo 702"

vendas_produto = dataset_unificado[
    dataset_unificado["name"] == produto_alvo
].copy()

if vendas_produto.empty:
    raise ValueError(f"O produto '{produto_alvo}' não foi encontrado.")

A análise deve considerar apenas esse produto. O filtro remove todos os demais itens do dataset unificado.
O .copy() cria uma cópia do resultado filtrado, evitando avisos e alterações acidentais no dataset original.

### 6. Transformação da data em mês

In [11]:
# ============================================================
# ETAPA 6 - Transformação da coluna de datas para o formato mensal
# ============================================================

vendas_produto["mes"] = (
    vendas_produto["created_at"]
    .dt.to_period("M")
    .dt.to_timestamp()
)


Como a previsão de demanda será realizada em base mensal, a coluna `created_at` foi convertida para uma referência única de mês.

A função `.dt.to_period("M")` remove o detalhe de dia e horário, mantendo apenas o ano e o mês de cada venda. Em seguida, `.dt.to_timestamp()` converte esse período novamente para data, utilizando o primeiro dia de cada mês como referência.

Por exemplo, vendas realizadas em qualquer dia de janeiro de 2026 passam a ser representadas por `2026-01-01`. Dessa forma, todos os registros do mesmo mês podem ser agrupados para calcular a demanda mensal do produto.

### 7. Demanda mensal

In [ ]:
# ============================================================
# ETAPA 7 - Cálculo da demanda mensal do produto-alvo
# A demanda corresponde à soma das unidades vendidas.
# ============================================================
vendas_produto["mes"] = vendas_produto["created_at"].dt.to_period("M").dt.to_timestamp()

demanda_mensal = (
    vendas_produto
    .groupby("mes")["quantity"]
    .sum()
    .sort_index()
)

# Inclusão explícita de meses sem venda com demanda igual a zero.
periodo_completo = pd.date_range(
    start=demanda_mensal.index.min(),
    end=demanda_mensal.index.max(),
    freq="MS"
)

demanda_mensal = demanda_mensal.reindex(periodo_completo, fill_value=0)
demanda_mensal.index.name = "mes"
demanda_mensal.name = "demanda_real"

print("\nDemanda mensal:")
display(demanda_mensal.to_frame())


Demanda mensal:


,demanda_real
mes,
2020-01-01,31
2020-02-01,16
2020-03-01,17
2020-04-01,32
2020-05-01,5
...,...
2026-08-01,25
2026-09-01,56
2026-10-01,29


As vendas são previstas em base mensal, não por dia.
```text 
Exemplos:
2026-01-03 → 2026-01-01
2026-01-28 → 2026-01-01
```

Todas as vendas de janeiro passam a pertencer ao mesmo mês: 2026-01-01.
Foi usada `created_at` como referência temporal. Se a regra de negócio definir que a venda deve ser reconhecida em `placed_at`, a substituição deve ser feita de forma consistente em todo o código.

Se um mês não teve vendas, ele pode não aparecer após o groupby. Para previsão de demanda, esse mês deve existir e ter valor 0; caso contrário, a média móvel pode ficar artificialmente alta.
freq="MS" significa “início de cada mês” (month start).

### 8. Separação entre treino e teste

In [12]:
# ============================================================
# ETAPA 8 - Separação entre treino e teste
# Treino: até dezembro de 2025.
# Teste: janeiro a março de 2026.
# ============================================================
fim_treino = pd.Timestamp("2025-12-01")
inicio_teste = pd.Timestamp("2026-01-01")
fim_teste = pd.Timestamp("2026-03-01")

treino = demanda_mensal[demanda_mensal.index <= fim_treino].copy()

teste = demanda_mensal[
    (demanda_mensal.index >= inicio_teste) &
    (demanda_mensal.index <= fim_teste)
].copy()

if len(teste) != 3:
    raise ValueError("O período de teste deve conter janeiro, fevereiro e março de 2026.")


O treino contem os dados até dezembro de 2025 e o teste contém janeiro, fevereiro e março de 2026. Utilizado para comparar a previsão com a demanda real. 

### 9. Média móvel de três meses + Atualização do histórico e ausência de leakage

In [13]:
# ============================================================
# ETAPA 9 - Previsão por média móvel de três meses
# Para cada mês, são utilizados somente dados anteriores
# à data prevista. Após cada mês, o valor real é incorporado
# ao histórico para a previsão do mês seguinte.
# ============================================================
historico = treino.copy()
previsoes = []
# historico começa apenas com os dados até dezembro de 2025. previsoes armazenará o resultado de cada mês.

for mes, demanda_real in teste.items():
    ultimos_tres_meses = historico[historico.index < mes].tail(3)

    if len(ultimos_tres_meses) < 3:
        raise ValueError(
            f"Não há histórico suficiente para prever o mês {mes.strftime('%Y-%m')}."
        )

    previsao = ultimos_tres_meses.mean()

    previsoes.append({
        "mes": mes,
        "demanda_real": demanda_real,
        "previsao": previsao
    })

    # O dado real do mês atual só entra no histórico
    # após sua própria previsão ter sido calculada.
    historico.loc[mes] = demanda_real


A previsão é feita antes de inserir o valor real do mês atual no histórico.
Assim, a venda real de janeiro não é usada para prever janeiro. Depois que janeiro termina e seu resultado se torna conhecido, ele pode ser utilizado na previsão de fevereiro. Isso representa uma previsão mensal atualizada ao longo do tempo, também chamada de validação walk-forward.

### 10. Erro absoluto e MAE

In [14]:
# ============================================================
# ETAPA 10 - Avaliação das previsões e cálculo do MAE
# ============================================================

# Converte a lista de previsões em uma tabela para facilitar
# a comparação entre demanda real, previsão e erro de cada mês.
resultado_previsao = pd.DataFrame(previsoes)

# Calcula a diferença absoluta entre a demanda observada e a prevista.
# O valor absoluto elimina o sinal: erros por excesso e por falta
# possuem o mesmo peso na avaliação.
resultado_previsao["erro_absoluto"] = (
    resultado_previsao["demanda_real"] -
    resultado_previsao["previsao"]
).abs()

# MAE (Mean Absolute Error) corresponde à média dos erros absolutos.
# Fórmula: MAE = soma dos erros absolutos / quantidade de períodos.
# Quanto menor o MAE, mais próximas as previsões estão da demanda real.
mae = resultado_previsao["erro_absoluto"].mean()

# Soma as previsões dos três meses para apresentar uma estimativa
# consolidada da demanda prevista para todo o primeiro trimestre.
soma_previsoes = resultado_previsao["previsao"].sum()

# Arredonda a previsão trimestral para uma unidade inteira, pois
# produtos físicos não podem ser adquiridos ou vendidos em frações.
soma_previsoes_arredondada = round(soma_previsoes)

# Arredonda as previsões mensais apenas para melhorar a apresentação.
# O cálculo do MAE foi realizado antes desse arredondamento.
resultado_previsao["previsao"] = resultado_previsao["previsao"].round(2)

# Exibe a comparação mensal entre demanda real, previsão e erro.
print("\nPrevisões para o primeiro trimestre de 2026:")
display(resultado_previsao)

# Apresenta o MAE e as previsões consolidadas do trimestre.
print(f"MAE: {mae:.2f} unidades")
print(f"Soma das previsões: {soma_previsoes:.2f} unidades")
print(f"Soma das previsões arredondada: {soma_previsoes_arredondada} unidades")


Previsões para o primeiro trimestre de 2026:


,mes,demanda_real,previsao,erro_absoluto
0,2026-01-01,79,38.67,40.333333
1,2026-02-01,68,53.67,14.333333
2,2026-03-01,60,56.33,3.666667


MAE: 19.44 unidades
Soma das previsões: 148.67 unidades
Soma das previsões arredondada: 149 unidades


Logo, o baseline errou, em média, aproximadamente 19 unidades por mês.

## Questão 6.2 - Validação

### Utilizando seu modelo treinado, qual é a soma total da previsão de vendas (arredondada para número inteiro) para o 'Bússola de Bordo 702' durante o primeiro trimestre de 2026?


Resposta: 149
Explicação:


A previsão foi realizada para a Bússola de Bordo 702 utilizando uma média móvel dos três meses anteriores.
| Mês | Demanda real | Previsão | Erro absoluto |
|---|---:|---:|---:|
| Janeiro/2026 | 79 | 38,67 | 40,33 |
| Fevereiro/2026 | 68 | 53,67 | 14,33 |
| Março/2026 | 60 | 56,33 | 3,67 |


A soma das previsões para o primeiro trimestre de 2026 foi de **148,67 unidades**, arredondada para **149 unidades**. A demanda real do período foi de 207 unidades.


## Questão 6.3 - Explique:

###  Como o baseline foi construído?

O baseline foi construído com uma média móvel simples de três meses. Para cada mês previsto, foi calculada a média das unidades vendidas nos três meses anteriores.

A previsão de janeiro de 2026 utilizou os dados de outubro, novembro e dezembro de 2025. Para fevereiro e março, foi aplicado um processo sequencial: a previsão de cada mês utilizou somente dados que já estavam disponíveis antes daquele mês.

### Como foi evitado o data leakage?

Foi evitado o uso de informações futuras. A demanda real do próprio mês previsto não foi utilizada no cálculo de sua previsão.

Por exemplo, a previsão de janeiro de 2026 foi calculada somente com dados até dezembro de 2025. Após o encerramento de janeiro, sua demanda real foi incorporada ao histórico para estimar fevereiro. Essa abordagem simula um processo de previsão mensal atualizado ao longo do tempo.

### O baseline é adequado para esse produto?

O baseline é útil como referência inicial por ser simples e interpretável. Entretanto, não se mostrou suficiente como único método para apoiar decisões críticas de compra e estoque.

As três previsões ficaram abaixo da demanda real, e o maior erro ocorreu em janeiro de 2026, quando foram previstas 38,67 unidades e foram vendidas 79 unidades. Esse comportamento indica risco de subestimar a demanda e contribuir para rupturas de estoque.

### Limitação do método

A média móvel utiliza apenas os três meses anteriores e não considera fatores como sazonalidade, tendência, campanhas comerciais, preço, disponibilidade de estoque ou características do produto. Por esse motivo, aumentos ou quedas abruptas na demanda podem não ser capturados adequadamente.